# 03 - Xây dựng & đánh giá mô hình học máy phân loại Priority

**Input:** `data/processed/redd_plus_cleaned.csv` (output của `01_data_cleaning.ipynb`)
**Output:** `data/processed/redd_predictions_for_gis.csv`, `outputs/models/best_model.joblib`, các bảng/biểu đồ trong `outputs/`

**Phương pháp:**
1. **Tiền xử lý trong `Pipeline`:** `OneHotEncoder` cho biến phân loại, `StandardScaler` cho biến số (cần cho Logistic Regression),
   đặc trưng cụm không gian KMeans (fit riêng từng tỉnh, chỉ trên tập train của mỗi fold → không rò rỉ dữ liệu test).
2. **Loại biến gây nhiễu / rò rỉ:** bỏ `status` (thông tin có sau khi xác minh, không liên quan Priority) và `year` (trùng với tỉnh).
3. **Đánh giá bằng `StratifiedGroupKFold` theo tiểu khu (`spatial_group`)**, so sánh với `StratifiedKFold` ngẫu nhiên để định lượng
   mức độ rò rỉ không gian.
4. **So sánh đầy đủ:** Dummy → Rule baseline (1 dòng) → Logistic Regression → Decision Tree → Random Forest → HistGradientBoosting.
5. **Ablation** các nhóm đặc trưng (lâm học vs vị trí) và **tinh chỉnh siêu tham số** trên chính chiến lược chia theo tiểu khu.
6. **Đánh giá riêng** rừng tự nhiên (High vs Medium), từng tỉnh, và khả năng **xếp hạng** các lô ưu tiên cao (PR-AUC, lift).
7. **Giải thích mô hình** bằng Permutation Importance trên cả 5 fold theo tiểu khu.
8. **Xuất dự đoán out-of-fold + xác suất cho toàn bộ điểm** cho ArcGIS / Power BI (notebook 04).

## 1. Thiết lập & đọc dữ liệu

In [ ]:
from pathlib import Path
import sys
import time
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, average_precision_score, precision_recall_curve,
)
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.model_utils import ProvinceKMeansFeatures, OriginRuleClassifier   # lớp tuỳ biến cho Pipeline

PROCESSED_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TABLE_DIR = ROOT / "outputs" / "tables"
MODEL_DIR = ROOT / "outputs" / "models"
for d in (PROCESSED_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})
PRIORITY_ORDER = ["High", "Medium", "Low"]
PRIORITY_COLORS = {"High": "#d7191c", "Medium": "#fdae61", "Low": "#1a9641"}
SEED = 42

df = pd.read_csv(PROCESSED_DIR / "redd_plus_cleaned.csv")
print(f"Số dòng: {len(df):,}  |  Số tiểu khu (nhóm không gian): {df['spatial_group'].nunique()}")
print(df["priority"].value_counts().reindex(PRIORITY_ORDER))

## 2. Chọn đặc trưng & tạo đặc trưng không gian

| Nhóm | Đặc trưng | Mã hoá |
|---|---|---|
| Lâm học | `forest_origin`, `land_cover_group`, `stock_level`, `land_type`, `stock_ord` | One-Hot / số thứ bậc |
| Quản lý | `owner_type` | One-Hot |
| Hành chính | `province`, `district` | One-Hot |
| Không gian | `latitude`, `longitude`, `dist_to_district_center_km`, cụm KMeans (`kmeans_cluster`, `kmeans_min_dist_km`) | Chuẩn hoá / One-Hot |

**Không đưa vào mô hình:** `status` (thông tin sau xác minh, Cramér's V ≈ 0,03), `year` (trùng với tỉnh), `id`/`plot`/`lot` (mã định danh),
`commune` (lặp thông tin vị trí). `spatial_group` (tiểu khu) chỉ dùng làm **nhóm chia CV**.

Hai lớp tuỳ biến được định nghĩa trong `src/model_utils.py` (để mô hình lưu bằng `joblib` nạp lại được ở nơi khác):
- `ProvinceKMeansFeatures`: chia mỗi tỉnh thành 6 cụm không gian bằng KMeans (trên toạ độ chiếu km), thêm mã cụm và khoảng cách
  tới tâm cụm gần nhất. Fit trong Pipeline → chỉ học từ tập train của mỗi fold.
- `OriginRuleClassifier`: baseline luật 1 dòng rút ra từ EDA (*Rừng tự nhiên → Medium, Rừng trồng → Low*).

In [ ]:
FORESTRY_CAT = ["forest_origin", "land_cover_group", "stock_level", "land_type", "owner_type"]
ADMIN_CAT = ["province", "district"]
SPATIAL_NUM = ["latitude", "longitude", "dist_to_district_center_km", "kmeans_min_dist_km"]

FEATURE_SETS = {
    # tên: (biến phân loại, biến số, có dùng KMeans không)
    "full":     (FORESTRY_CAT + ADMIN_CAT + ["kmeans_cluster"], ["stock_ord"] + SPATIAL_NUM, True),
    "no_coord": (FORESTRY_CAT + ADMIN_CAT, ["stock_ord"], False),
}
RAW_FEATURES = FORESTRY_CAT + ADMIN_CAT + ["stock_ord", "latitude", "longitude", "dist_to_district_center_km"]

X = df[RAW_FEATURES].copy()
y = df["priority"].values
groups = df["spatial_group"].values
NAT_MASK = (df["forest_origin"] == "Natural forest").values


def make_pipeline(clf, feature_set: str = "full", scale_numeric: bool = True, dense: bool = False,
                  n_clusters: int = 6):
    cats, nums, use_kmeans = FEATURE_SETS[feature_set]
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=not dense), cats),
        ("num", StandardScaler() if scale_numeric else "passthrough", nums),
    ])
    steps = [("kmeans", ProvinceKMeansFeatures(n_clusters=n_clusters))] if use_kmeans else []
    return Pipeline(steps + [("pre", pre), ("clf", clf)])

## 3. Chiến lược chia dữ liệu: Ngẫu nhiên vs Theo tiểu khu

- **`StratifiedKFold` (5-fold ngẫu nhiên):** điểm cùng tiểu khu vừa ở train vừa ở test → đo khả năng **nội suy** trong vùng đã khảo sát.
- **`StratifiedGroupKFold` (5-fold theo `spatial_group`):** cả tiểu khu chỉ nằm ở train *hoặc* test → đo khả năng **ngoại suy sang tiểu khu mới**.
  Đây là cách đánh giá **chính** của đồ án vì phản ánh đúng tình huống sử dụng thực tế (xếp hạng lô rừng ở khu vực chưa phân loại).

In [ ]:
cv_random = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_group = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
splits_random = list(cv_random.split(X, y))
splits_group = list(cv_group.split(X, y, groups))

for fold, (tr, te) in enumerate(splits_group, 1):
    shared = set(groups[tr]) & set(groups[te])
    dist = pd.Series(y[te]).value_counts(normalize=True).reindex(PRIORITY_ORDER).round(3).to_dict()
    print(f"Fold {fold}: n_test={len(te):5,} | tiểu khu trùng train-test={len(shared)} | tỷ lệ lớp={dist}")

## 4. So sánh các mô hình (5-fold CV)

Mọi mô hình học máy đều dùng `class_weight="balanced"` để xử lý mất cân bằng lớp (High chỉ ~17%), giúp so sánh công bằng.
Chỉ số chính: **F1-macro** (trung bình F1 của 3 lớp) và **F1 lớp High** — lớp quan trọng nhất với công tác quản lý.

In [ ]:
def cv_oof_predict(model, X, y, splits):
    """Dự đoán out-of-fold (nhãn + xác suất theo thứ tự PRIORITY_ORDER)."""
    preds = np.empty(len(y), dtype=object)
    proba = np.zeros((len(y), len(PRIORITY_ORDER)))
    for tr, te in splits:
        m = clone(model).fit(X.iloc[tr], y[tr])
        preds[te] = m.predict(X.iloc[te])
        p = m.predict_proba(X.iloc[te])
        for j, c in enumerate(m.classes_):
            proba[te, PRIORITY_ORDER.index(c)] = p[:, j]
    return preds, proba


def eval_predictions(y_true, y_pred, nat_mask=NAT_MASK):
    hm = nat_mask & (y_true != "Low")            # rừng tự nhiên, bài toán High vs Medium
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall lớp High": recall_score(y_true == "High", y_pred == "High", zero_division=0),
        "F1 lớp High": f1_score(y_true == "High", y_pred == "High", zero_division=0),
        "F1 High (rừng tự nhiên)": (f1_score(y_true[hm] == "High", y_pred[hm] == "High", zero_division=0)
                                    if hm.any() else np.nan),
    }


models = {
    "0. Dummy (luôn đoán Medium)": DummyClassifier(strategy="most_frequent"),
    "1. Rule baseline (Tự nhiên→Med, Trồng→Low)": OriginRuleClassifier(),
    "2. Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0)),
    "3. Decision Tree": make_pipeline(
        DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, class_weight="balanced", random_state=SEED),
        scale_numeric=False),
    "4. Random Forest": make_pipeline(
        RandomForestClassifier(n_estimators=300, max_depth=16, min_samples_leaf=2, class_weight="balanced",
                               n_jobs=-1, random_state=SEED), scale_numeric=False),
    "5. HistGradientBoosting": make_pipeline(
        HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=20,
                                       class_weight="balanced", random_state=SEED),
        scale_numeric=False, dense=True),
}

rows, oof = [], {}          # oof[tên] = (nhãn, xác suất) theo GroupKFold
for name, model in models.items():
    t0 = time.time()
    pred_r, _ = cv_oof_predict(model, X, y, splits_random)
    pred_g, prob_g = cv_oof_predict(model, X, y, splits_group)
    oof[name] = (pred_g, prob_g)
    rows.append({"Mô hình": name, "Chia CV": "Random 5-fold", **eval_predictions(y, pred_r)})
    rows.append({"Mô hình": name, "Chia CV": "GroupKFold (tiểu khu)", **eval_predictions(y, pred_g)})
    print(f"{name:45s} xong sau {time.time() - t0:5.1f}s")

results_df = pd.DataFrame(rows).round(4)
results_df.to_csv(TABLE_DIR / "model_comparison_cv.csv", index=False, encoding="utf-8-sig")
results_df[results_df["Chia CV"] == "GroupKFold (tiểu khu)"].drop(columns="Chia CV").set_index("Mô hình")

In [ ]:
piv = results_df.pivot(index="Mô hình", columns="Chia CV", values="F1 (macro)")[["Random 5-fold", "GroupKFold (tiểu khu)"]]
piv["Chênh lệch (Random − Group)"] = (piv["Random 5-fold"] - piv["GroupKFold (tiểu khu)"]).round(4)
display(piv)

ax = piv[["Random 5-fold", "GroupKFold (tiểu khu)"]].plot(kind="barh", figsize=(9, 4.5), color=["#2b83ba", "#fdae61"], width=0.75)
ax.set_xlabel("F1-score (macro)"); ax.set_ylabel("")
ax.set_title("F1-macro: chia ngẫu nhiên vs chia theo tiểu khu")
ax.set_xlim(0, 0.9); ax.invert_yaxis(); ax.legend(loc="lower right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8.5)
plt.tight_layout(); plt.savefig(FIG_DIR / "model_comparison_f1.png", bbox_inches="tight"); plt.show()

__RESULTS_MD__

## 5. Ablation: mô hình dự đoán nhờ "loại rừng" hay nhờ "nhớ vị trí"?

Cùng một thuật toán (Logistic Regression — đơn giản, ổn định), chỉ thay đổi tập đặc trưng, đánh giá bằng GroupKFold theo tiểu khu.

In [ ]:
ABLATION = {
    "A. Chỉ nguồn gốc rừng":               (["forest_origin"], [], False),
    "B. Chỉ vị trí (tỉnh, huyện, toạ độ, cụm KMeans)": (ADMIN_CAT + ["kmeans_cluster"], SPATIAL_NUM, True),
    "C. Lâm học + chủ rừng (không vị trí)": (FORESTRY_CAT, ["stock_ord"], False),
    "D. Lâm học + chủ rừng + tỉnh/huyện":   FEATURE_SETS["no_coord"],
    "E. Đầy đủ (D + toạ độ + cụm KMeans)":  FEATURE_SETS["full"],
}

ablation_rows = []
for label, fs in ABLATION.items():
    FEATURE_SETS["_tmp"] = fs
    pipe = make_pipeline(LogisticRegression(max_iter=3000, class_weight="balanced"), feature_set="_tmp")
    pred_g, _ = cv_oof_predict(pipe, X, y, splits_group)
    ablation_rows.append({"Tập đặc trưng": label, **eval_predictions(y, pred_g)})
FEATURE_SETS.pop("_tmp")

ablation_df = pd.DataFrame(ablation_rows).round(4)
ablation_df.to_csv(TABLE_DIR / "feature_ablation.csv", index=False, encoding="utf-8-sig")
ablation_df.set_index("Tập đặc trưng")

__ABLATION_MD__

## 6. Tinh chỉnh siêu tham số (theo GroupKFold tiểu khu)

Thử một lưới nhỏ các cấu hình tiêu biểu cho 3 thuật toán mạnh nhất, với cả 2 tập đặc trưng (`full` và `no_coord`).
Mô hình cuối cùng là cấu hình có **F1-macro (GroupKFold) cao nhất**.

> Lưu ý: chọn cấu hình trên cùng các fold dùng để báo cáo sẽ hơi lạc quan (selection bias). Với lưới nhỏ (~12 cấu hình)
> và chênh lệch giữa các cấu hình rất nhỏ, sai lệch này không đáng kể; cách chặt chẽ hơn là *nested cross-validation*.

In [ ]:
candidates = {}
for fs in ["full", "no_coord"]:
    for C in [0.1, 0.3, 1.0]:
        candidates[f"LR | C={C} | {fs}"] = make_pipeline(
            LogisticRegression(max_iter=3000, class_weight="balanced", C=C), feature_set=fs)
    candidates[f"HGB | lr=0.05, leaf=15, l2=1 | {fs}"] = make_pipeline(
        HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=30,
                                       l2_regularization=1.0, class_weight="balanced", random_state=SEED),
        feature_set=fs, scale_numeric=False, dense=True)
    candidates[f"RF | depth=12, leaf=5 | {fs}"] = make_pipeline(
        RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5, max_features="sqrt",
                               class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
        feature_set=fs, scale_numeric=False)

tuning_rows = []
for name, pipe in candidates.items():
    t0 = time.time()
    pred_c, proba_c = cv_oof_predict(pipe, X, y, splits_group)
    oof[name] = (pred_c, proba_c)
    m = eval_predictions(y, pred_c)
    tuning_rows.append({"Cấu hình": name, "F1 (macro)": m["F1 (macro)"], "F1 lớp High": m["F1 lớp High"],
                        "Accuracy": m["Accuracy"], "Thời gian (s)": round(time.time() - t0, 1)})

# Gộp với các mô hình ở mục 4 để chọn mô hình tốt nhất trên toàn bộ ứng viên
for name in ["2. Logistic Regression", "3. Decision Tree", "4. Random Forest", "5. HistGradientBoosting"]:
    m = eval_predictions(y, oof[name][0])
    tuning_rows.append({"Cấu hình": name + " (mục 4) | full", "F1 (macro)": m["F1 (macro)"],
                        "F1 lớp High": m["F1 lớp High"], "Accuracy": m["Accuracy"], "Thời gian (s)": np.nan})

tuning_df = pd.DataFrame(tuning_rows).sort_values("F1 (macro)", ascending=False).round(4).reset_index(drop=True)
tuning_df.to_csv(TABLE_DIR / "hyperparameter_tuning.csv", index=False, encoding="utf-8-sig")
tuning_df

In [ ]:
all_pipes = {**candidates, **{k + " (mục 4) | full": v for k, v in models.items()}}
BEST_NAME = tuning_df.loc[0, "Cấu hình"]
best_model = all_pipes[BEST_NAME]
oof_key = BEST_NAME.replace(" (mục 4) | full", "")
y_oof_pred, y_oof_proba = oof[oof_key]

print(f">>> Mô hình được chọn: {BEST_NAME}")
print(f"    F1-macro (GroupKFold) = {tuning_df.loc[0, 'F1 (macro)']:.4f}")
print(f"    Chênh lệch với cấu hình thứ 2: {tuning_df.loc[0, 'F1 (macro)'] - tuning_df.loc[1, 'F1 (macro)']:.4f}")

## 7. Đánh giá chi tiết mô hình được chọn

### 7.1 Classification report & ma trận nhầm lẫn (dự đoán out-of-fold theo tiểu khu)

In [ ]:
print(classification_report(y, y_oof_pred, labels=PRIORITY_ORDER, digits=4))

cm = confusion_matrix(y, y_oof_pred, labels=PRIORITY_ORDER)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, mat, is_count, title in [(axes[0], cm, True, "Số điểm"), (axes[1], cm_pct, False, "% theo nhãn thực tế (Recall)")]:
    ax.imshow(mat, cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels(PRIORITY_ORDER)
    ax.set_yticks(range(3)); ax.set_yticklabels(PRIORITY_ORDER)
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế"); ax.set_title(f"Ma trận nhầm lẫn — {title}")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{mat[i, j]:,}" if is_count else f"{mat[i, j]:.1f}%", ha="center", va="center",
                    color="white" if mat[i, j] > mat.max() / 2 else "black", fontweight="bold")
plt.suptitle(BEST_NAME, fontsize=10)
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix.png", bbox_inches="tight"); plt.show()

### 7.2 Theo nhóm: rừng tự nhiên (High vs Medium) và từng tỉnh

In [ ]:
segments = {
    "Toàn bộ": np.ones(len(df), bool),
    "Rừng tự nhiên": NAT_MASK,
    "Rừng trồng / đất chưa có rừng": ~NAT_MASK,
    "Lâm Đồng": (df["province"] == "Lam Dong").values,
    "Nghệ An": (df["province"] == "Nghe An").values,
}
seg_rows = []
for seg, m in segments.items():
    r = eval_predictions(y[m], y_oof_pred[m], nat_mask=NAT_MASK[m])
    seg_rows.append({"Nhóm": seg, "Số điểm": int(m.sum()), "Tỷ lệ High thực tế": (y[m] == "High").mean(),
                     **{k: r[k] for k in ["Accuracy", "F1 (macro)", "Recall lớp High", "F1 lớp High"]}})
seg_df = pd.DataFrame(seg_rows).round(4)
seg_df.to_csv(TABLE_DIR / "best_model_by_segment.csv", index=False, encoding="utf-8-sig")
seg_df

### 7.3 Khả năng xếp hạng lô ưu tiên cao (ứng dụng thực tế)

Trong thực tế, cơ quan quản lý thường cần **danh sách lô cần ưu tiên kiểm tra trước** với nguồn lực có hạn.
Thay vì nhãn cứng, dùng xác suất `prob_high` để xếp hạng: nếu chỉ kiểm tra *x%* số lô có `prob_high` cao nhất thì phát hiện được
bao nhiêu % lô High thực tế? (so với chọn ngẫu nhiên chỉ được *x%*).

In [ ]:
prob_high = y_oof_proba[:, PRIORITY_ORDER.index("High")]
is_high = (y == "High").astype(int)
ap = average_precision_score(is_high, prob_high)
print(f"PR-AUC (Average Precision) lớp High: {ap:.3f}   (ngẫu nhiên = {is_high.mean():.3f})")

order = np.argsort(-prob_high)
lift_rows = []
for top in [0.05, 0.10, 0.20, 0.30, 0.50]:
    k = int(round(top * len(y)))
    captured = is_high[order[:k]].sum() / is_high.sum()
    lift_rows.append({"Kiểm tra top % lô": f"{top:.0%}", "Số lô": k,
                      "% lô High phát hiện được": round(captured * 100, 1),
                      "Precision trong nhóm": round(is_high[order[:k]].mean(), 3),
                      "Lift so với ngẫu nhiên": round(captured / top, 2)})
lift_df = pd.DataFrame(lift_rows)
lift_df.to_csv(TABLE_DIR / "best_model_lift_high.csv", index=False, encoding="utf-8-sig")
display(lift_df)

prec, rec, _ = precision_recall_curve(is_high, prob_high)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
axes[0].plot(rec, prec, color=PRIORITY_COLORS["High"])
axes[0].axhline(is_high.mean(), ls="--", c="grey", label=f"Ngẫu nhiên ({is_high.mean():.2f})")
axes[0].set_xlabel("Recall (High)"); axes[0].set_ylabel("Precision (High)")
axes[0].set_title(f"Đường Precision–Recall lớp High (AP = {ap:.3f})"); axes[0].legend()
frac = np.arange(1, len(y) + 1) / len(y)
axes[1].plot(frac * 100, np.cumsum(is_high[order]) / is_high.sum() * 100, color=PRIORITY_COLORS["High"], label="Mô hình")
axes[1].plot([0, 100], [0, 100], ls="--", c="grey", label="Ngẫu nhiên")
axes[1].set_xlabel("% số lô được kiểm tra (xếp theo prob_high giảm dần)"); axes[1].set_ylabel("% lô High phát hiện được")
axes[1].set_title("Đường tích luỹ (gain chart)"); axes[1].legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "best_model_pr_gain_high.png", bbox_inches="tight"); plt.show()

## 8. Giải thích mô hình: Permutation Feature Importance

Xáo trộn từng đặc trưng gốc trên **tập test của cả 5 fold theo tiểu khu** và đo mức giảm F1-macro. Cách này không thiên vị
biến liên tục/biến nhiều giá trị như *impurity importance* của cây. Các đặc trưng phái sinh (`stock_level`, `land_cover_group`...)
tương quan mạnh với `land_type` nên tầm quan trọng bị chia sẻ — cần đọc theo **nhóm** thay vì từng biến riêng lẻ.

In [ ]:
rng = np.random.default_rng(SEED)
used = [c for c in RAW_FEATURES if c in X.columns]
drops = {c: [] for c in used}
for tr, te in splits_group:
    m = clone(best_model).fit(X.iloc[tr], y[tr])
    Xte, yte = X.iloc[te], y[te]
    base = f1_score(yte, m.predict(Xte), average="macro")
    for col in used:
        for _ in range(3):
            Xp = Xte.copy()
            Xp[col] = rng.permutation(Xp[col].values)
            drops[col].append(base - f1_score(yte, m.predict(Xp), average="macro"))

imp_df = pd.DataFrame({"mean": {c: np.mean(v) for c, v in drops.items()},
                       "std": {c: np.std(v) for c, v in drops.items()}}).sort_values("mean")
imp_df.to_csv(TABLE_DIR / "permutation_importance.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.barh(imp_df.index, imp_df["mean"], xerr=imp_df["std"], color="#2b83ba", capsize=3)
ax.axvline(0, c="grey", lw=0.8)
ax.set_xlabel("Mức giảm F1-macro khi xáo trộn đặc trưng")
ax.set_title(f"Permutation Importance (GroupKFold, 5 fold)\n{BEST_NAME}", fontsize=10)
plt.tight_layout(); plt.savefig(FIG_DIR / "feature_importance.png", bbox_inches="tight"); plt.show()
imp_df.sort_values("mean", ascending=False).round(4)

__IMPORTANCE_MD__

## 9. Xuất kết quả cho ArcGIS & Power BI, lưu mô hình

Xuất **toàn bộ điểm** kèm:
- `predicted_priority`: nhãn dự đoán **out-of-fold** — mỗi điểm được dự đoán bởi mô hình **không học** trên tiểu khu của điểm đó.
- `prob_high`, `prob_medium`, `prob_low`, `confidence`: xác suất từng mức và độ tự tin.
- `priority_rank_high`: thứ hạng theo `prob_high` (1 = cần ưu tiên nhất) — dùng làm danh sách kiểm tra.
- `is_correct`, `error_type`: phục vụ phân tích phần dư không gian (notebook 04) và lọc trên Power BI.

Mô hình cuối cùng được **huấn luyện lại trên toàn bộ dữ liệu** và lưu bằng `joblib` để dùng dự đoán lô rừng mới.

In [ ]:
gis_df = df[["id", "latitude", "longitude", "province", "district", "commune", "sub_region", "spatial_group",
             "owner_type", "forest_origin", "land_cover_group", "stock_level", "stock_ord", "land_type",
             "date", "year", "month", "status", "low_precision_coord", "priority"]].copy()
gis_df["predicted_priority"] = y_oof_pred
for j, c in enumerate(PRIORITY_ORDER):
    gis_df[f"prob_{c.lower()}"] = y_oof_proba[:, j].round(4)
gis_df["confidence"] = y_oof_proba.max(axis=1).round(4)
gis_df["priority_rank_high"] = gis_df["prob_high"].rank(ascending=False, method="first").astype(int)
gis_df["is_correct"] = (gis_df["priority"] == gis_df["predicted_priority"]).astype(int)
gis_df["error_type"] = np.where(gis_df["is_correct"] == 1, "Correct",
                                gis_df["priority"] + " -> " + gis_df["predicted_priority"])

out_csv = PROCESSED_DIR / "redd_predictions_for_gis.csv"
gis_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {out_csv.relative_to(ROOT)}  —  {len(gis_df):,} dòng × {gis_df.shape[1]} cột")
print(gis_df["error_type"].value_counts().to_string())

final_model = clone(best_model).fit(X, y)
joblib.dump({"model": final_model, "features": RAW_FEATURES, "name": BEST_NAME, "classes": PRIORITY_ORDER},
            MODEL_DIR / "best_model.joblib")
print(f"\nĐã lưu mô hình: {(MODEL_DIR / 'best_model.joblib').relative_to(ROOT)}")

__CONCLUSION_MD__